# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List RecordSet @ids, their fields and columns
print("Available record sets:")
record_set_ids = [rs['@id'] for rs in dataset._metadata_dict.get('recordSet', [])]
if record_set_ids:
    for rs in dataset._metadata_dict['recordSet']:
        rs_id = rs['@id']
        print(f"\nRecord Set @id: {rs_id}")
        print(f"  Name: {rs.get('name', '<no name>')}")
        field_ids = [f['@id'] for f in rs.get('field', [])] if isinstance(rs.get('field', []), list) else [rs['field']['@id']] if rs.get('field') else []
        print(f"  Fields: {field_ids}")
        # For each field, print columns
        for field in rs.get('field', []):
            print(f"    Field @id: {field['@id']}")
            if 'column' in field:
                if isinstance(field['column'], list):
                    cols = [c['@id'] for c in field['column']]
                else:
                    cols = [field['column']['@id']]
                print(f"      Columns: {cols}")
else:
    print("No inline record sets found in metadata. Attempting to list via dataset.record_set_ids property...")
    # Newer mlcroissant API
    if hasattr(dataset, 'record_set_ids'):
        for rs_id in dataset.record_set_ids:
            print(f"  {rs_id}")
    else:
        print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List discovered record sets
croissant_dict = dataset._metadata_dict

# Find all record sets by their @id
record_sets = []
if 'recordSet' in croissant_dict and isinstance(croissant_dict['recordSet'], list):
    for rs in croissant_dict['recordSet']:
        record_sets.append(rs['@id'])

# If none (shouldn't happen), try .record_set_ids
if not record_sets and hasattr(dataset, 'record_set_ids'):
    record_sets = list(dataset.record_set_ids)

print('Found record sets:', record_sets)

dataframes = {}
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} rows from {record_set}")
    except Exception as e:
        print(f"Could not load records for {record_set}: {e}")

# For demonstration, select the first record set if any
example_rs = record_sets[0] if record_sets else None
if example_rs and example_rs in dataframes:
    print(f"Columns in {example_rs}: {dataframes[example_rs].columns.tolist()}")
    display(dataframes[example_rs].head())
else:
    print("No record sets loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# For this dataset, let's infer a numeric field
numeric_field_candidates = []
if example_rs and example_rs in dataframes:
    df = dataframes[example_rs]
    for col in df.columns:
        try:
            # Check if conversion to numeric yields any non-NaN
            if pd.to_numeric(df[col], errors='coerce').notna().sum() > 0:
                numeric_field_candidates.append(col)
        except Exception:
            pass
    if numeric_field_candidates:
        print("Numeric field candidates:", numeric_field_candidates)
        numeric_field = numeric_field_candidates[0]
    else:
        print("No numeric fields found.")
        numeric_field = None
else:
    print("No data frame to analyze.")

# Apply filter and normalization if numeric field exists
if example_rs and example_rs in dataframes and numeric_field:
    df = dataframes[example_rs].copy()
    # Ensure numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by another field if possible
    group_fields = [c for c in df.columns if c != numeric_field]
    group_field = group_fields[0] if group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable field for grouping found.")
else:
    print("Skipping EDA - no numeric field or DataFrame available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_rs and example_rs in dataframes and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[example_rs][numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if len(dataframes[example_rs].columns) > 1:
        other_field = [col for col in dataframes[example_rs].columns if col != numeric_field][0]
        plt.figure(figsize=(10,4))
        sns.boxplot(x=other_field, y=numeric_field, data=dataframes[example_rs])
        plt.title(f"{numeric_field} by {other_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field or DataFrame available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded FAIR² Croissant metadata describing the extracellular vesicle proteomics dataset using mass spectrometry from human mast cells.
* Record sets, fields, and columns were dynamically discovered and referenced by their `@id` values.
* We extracted available records for analysis, automatically selected and normalized a numeric field, and visualized its distribution.
* You may further tailor filtering, feature engineering, and visualization using specific `@id` values for fields and columns, referencing this workflow as a guide for `mlcroissant`-based FAIR datasets.